# CLMET Productivity Workflow (1710-1780)

This notebook demonstrates a CLMET workflow using the current MorphAnalyzer API:
1. Read corpus files from `data/clmet/1710-1780` with tmtoolkit
2. Tokenize with spaCy and build analyzer-ready counts automatically
3. Run a preview sample first
4. Choose a narrow analysis scope before any longer run
5. Inspect hapax-based productivity and targeted derivational slices

In [ ]:
from pathlib import Path
import time

import polars as pl
import spacy

from morph_parser import (
    MorphAnalyzer,
    MorphParser,
 )
from morph_parser.corpus import CorpusProcessor, get_text_paths, readtext

In [ ]:
text_paths = get_text_paths("data/clmet/1710-1780", recursive=False)
corpus = readtext(text_paths)

In [ ]:
# Option 3: Maximize speed (disable more components)
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

In [ ]:
processor = CorpusProcessor()
df_tokens = processor.process_corpus(corpus, nlp_model=nlp, n_process=4, lemma=False)

In [ ]:
print(f"Token rows: {df_tokens.height:,}")
print(f"Unique token types: {df_tokens.select(pl.col('token').n_unique()).item():,}")

In [ ]:
parser = MorphParser.from_default(device="cpu", decompose="hybrid")
analyzer = MorphAnalyzer.from_spacy_tokens(df_tokens, parser)

print("Analyzer ready.")
print("Detected POS tags:", analyzer.available_pos())

In [ ]:
t0 = time.perf_counter()
preview_bundle = analyzer.preview(sample_size=1000, min_token_length=4, random_seed=42)
preview_run_time = time.perf_counter() - t0

preview_summary = preview_bundle.morpheme_summary.sort("N", descending=True)
preview_final_segment_summary = (
    analyzer.final_segment_summary(
        family="derivational",
        min_segment_count=2,
    )
    .filter(pl.col("V") > 1)
    .select([
        "morpheme",
        "N",
        "V",
        "V1",
        "F",
        "Y",
        "I",
        "P_baayen",
        "S_baayen",
        "H",
        "H_norm",
        "dominant_morpheme_family",
        "dominant_morpheme_function",
        "dominant_role_confidence",
    ])
    .sort("N", descending=True)
)
preview_diagnostics = preview_bundle.diagnostics

print(f"Preview runtime: {preview_run_time:.2f}s")
print("Preview diagnostics:")
print(preview_diagnostics)

In [ ]:
pl.Config.set_tbl_cols(-1)
print("Preview final-segment derivational summary:")
preview_final_segment_summary.head(20)

## How To Read The Preview Summary

This first table is a quick suffix-oriented scan of the preview sample. It is meant to surface patterns worth following up in a longer run, not to settle interpretation on its own.

Useful columns:

- `N`: total token count contributed by this morpheme in the preview sample.
- `V`: number of distinct word types containing the morpheme in this final-segment slice.
- `V1`: number of hapax types, i.e. types that occur once in the sample.
- `F`, `Y`, `I`: diversity and productivity summaries. Read them comparatively across rows rather than as standalone verdicts.
- `P_baayen`: hapax-based productivity, useful for asking whether a suffix appears to be recruiting new types in the sample.
- `S_baayen`: expansion potential relative to the observed vocabulary in the sample.
- `H` and `H_norm`: entropy-based dispersion measures that help distinguish concentrated rows from rows spread across many types.
- `dominant_morpheme_family`, `dominant_morpheme_function`, and `dominant_role_confidence`: the current best role label inferred from the supporting token evidence.

Questions this table can raise:

1. Which suffixes look robust even in a small sample?
2. Which rows are frequent but structurally mixed or low-confidence?
3. Which rows deserve token-level inspection before a longer run?

In [ ]:
preview_support_rows = ["ion", "ness", "ive", "ical"]

preview_support_summary = analyzer.morpheme_support_for_rows(
    preview_support_rows,
    family="derivational",
    final_segment_only=True,
    min_segment_count=2,
    top_n=5,
)

print("Preview support for historically or structurally informative rows:")
display(preview_support_summary)

## Inspect Preview Support

The table above adds token-level support for the preview suffix view so rows can be checked against actual evidence: example tokens, example segmentations, and common bases in the preview sample.

Useful columns:

- `N` and `V`: how much token and type support the row has in the preview sample.
- `example_tokens`: concrete lexical items contributing to the row.
- `example_segmentations`: the current segmentations for those example items.
- `top_bases` and `top_base_frequencies`: which stems are driving the row most strongly in the sample.

In this corpus, historically valid variation should not be over-determined away. Rows such as `-or` and `-ent` can reflect legitimate derivational patterns, including historical alternation with other agentive or learned endings. Use this support table to review segmentation evidence, not to assume that unfamiliar historical forms are errors.

Questions this table can raise:

1. Is this row supported by a coherent set of tokens, or is it structurally mixed?
2. Are the apparent examples historically plausible, or do they point to a preprocessing or segmentation artifact?
3. Does the row belong in the suffix-oriented view, or is it better treated as a separate structural case?

## Prepare Reusable Segments And Choose A Narrow Run Scope

`analyzer.run()` is expensive when it has to parse a large new vocabulary slice. For repeated analyst work, a better pattern is to parse once and then reuse that segmented inventory for multiple scoped summaries.

This chunk now does two things:

1. It materializes a reusable segmented token table with role labels.
2. It runs the first scoped analysis from that prepared inventory rather than reparsing the noun slice from scratch.

Why this helps:

- the expensive step is token segmentation, not the summary table itself;
- once segments are prepared, later scoped runs can reuse them;
- this avoids a brittle list-first shortcut that could miss relevant cases such as embedded `ize` in `globalization` or alternants like `ise` and `ize`.

Questions this step answers:

1. What exactly am I about to spend time computing?
2. Am I paying the parsing cost once or every time I change scope?
3. Which POS and morpheme families are in scope for the first full summary?

In [ ]:
analysis_pos = ["NOUN"]
analysis_families = ["derivational"]

print("Explicit analysis scope:")
print({"pos": analysis_pos, "morpheme_families": analysis_families})

t0 = time.perf_counter()
segmented_tokens = analyzer.prepare_segments(annotate_roles=True)
segment_prep_time = time.perf_counter() - t0

print(f"Prepared segmented inventory runtime: {segment_prep_time:.2f}s")
print(f"Prepared segmented rows: {segmented_tokens.height:,}")

t0 = time.perf_counter()
bundle = analyzer.run_from_segments(
    pos=analysis_pos,
    morpheme_families=analysis_families,
 )
scoped_run_time = time.perf_counter() - t0

print(f"Scoped analyzer runtime from prepared segments: {scoped_run_time:.2f}s")
print("Scoped diagnostics:")
print(bundle.diagnostics)

## Inspect The Narrow Scoped Run

`analyzer.run_from_segments(...)` returns an analysis bundle from the prepared segmented inventory. The expensive parsing work has already happened in the previous cell, so this step is now focused on summarization and inspection rather than re-segmentation.

What each output is doing:

1. `final_segment_summary`: the main suffix-oriented productivity table for final derivational segments.
2. `target_summary`: a focused slice for a small set of suffixes we want to compare directly.
3. `preview_target_summary`: the matching preview slice, useful for checking whether the small-sample picture is broadly consistent with the scoped run.
4. `hapax_summary`: a ranked view of rows with hapax evidence, useful for asking which suffixes show signs of continuing type expansion.
5. `non_final_derivational`: structurally non-final derivational material that should be inspected separately from the default suffix table.
6. `bases_for_morpheme("ness")`: a concrete base table showing which stems contribute most to one suffix of interest.

This keeps items like `after` or `man` out of the default suffix table without hiding them from inspection.

As you read the outputs, the main question is not just which rows are frequent. It is also whether the row is structurally coherent, how strongly it is supported, and whether preview and scoped-run evidence tell the same story. The practical advantage of this workflow is that new scoped queries can now reuse the prepared segment inventory instead of paying the full parsing cost each time.

In [ ]:
usage = bundle.morpheme_usage
summary = bundle.morpheme_summary.sort("N", descending=True)
base_pairs = bundle.base_affix_pairs
diagnostics = bundle.diagnostics

TARGET_SUFFIXES = ["ness", "ity", "age", "ment", "ly"]

final_segment_summary = (
    analyzer.final_segment_summary(
        family="derivational",
        min_segment_count=2,
    )
    .filter(pl.col("V") > 1)
    .select([
        "morpheme",
        "N",
        "V",
        "V1",
        "F",
        "Y",
        "I",
        "P_baayen",
        "S_baayen",
        "H",
        "H_norm",
        "dominant_morpheme_family",
        "dominant_morpheme_function",
        "dominant_role_confidence",
    ])
    .sort("N", descending=True)
)

non_final_derivational = (
    analyzer.non_final_segment_summary(
        family="derivational",
        min_segment_count=2,
        top_n_examples=5,
    )
    .select([
        "morpheme",
        "N",
        "V",
        "example_tokens",
        "example_segmentations",
        "dominant_morpheme_family",
        "dominant_morpheme_function",
        "dominant_role_confidence",
    ])
    .sort("N", descending=True)
)

print("Final-segment derivational summary from the narrow scoped run:")
display(final_segment_summary.head(20))

print("Check whether -ly is recognized as a derivational final segment:")
display(final_segment_summary.filter(pl.col("morpheme") == "ly"))

target_summary = (
    final_segment_summary
    .filter(pl.col("morpheme").is_in(TARGET_SUFFIXES))
    .sort("N", descending=True)
)

preview_target_summary = (
    preview_final_segment_summary
    .filter(pl.col("morpheme").is_in(TARGET_SUFFIXES))
    .sort("N", descending=True)
)

hapax_summary = (
    final_segment_summary
    .filter(pl.col("V1") > 0)
    .select([
        "morpheme",
        "N",
        "V",
        "V1",
        "P_baayen",
        "S_baayen",
        "H",
        "H_norm",
    ])
    .sort("P_baayen", descending=True)
    .head(20)
)

print("Target derivational suffix summary from the narrow scoped run:")
display(target_summary)

print("Preview summary for the same target suffixes:")
display(preview_target_summary)

print("Baayen-style hapax productivity leaders among final derivational segments:")
display(hapax_summary)

print("Non-final derivational rows to inspect separately:")
display(non_final_derivational.head(20))

print("Top bases for -ness from the narrow scoped run:")
display(analyzer.bases_for_morpheme("ness", top_n=20))

In [ ]:
output_dir = Path("_quarto/data/clmet_productivity_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

usage.write_parquet(output_dir / "usage.parquet")
summary.write_parquet(output_dir / "summary.parquet")
base_pairs.write_parquet(output_dir / "base_affix_pairs.parquet")
target_summary.write_csv(output_dir / "target_suffix_productivity.csv")
pl.DataFrame([diagnostics]).write_csv(output_dir / "run_diagnostics.csv")

output_dir